# Introdução ao Credit Risk+

## O Modelo Credit Risk+

**Credit Risk+** é um modelo estatístico desenvolvido pelo **Credit Suisse First Boston** (1997) para medir o risco de crédito em um portfólio de exposições. Diferentemente de modelos baseados em simulação (como Monte Carlo), o Credit Risk+ usa técnicas analíticas provenientes da teoria de seguros para calcular a distribuição completa de perdas por default.

### Características Principais

1. **Abordagem Não-Paramétrica**: Não faz suposições sobre as *causas* do default, apenas observa que acontecem
2. **Distribuição Exata**: Calcula analiticamente a distribuição de perdas via **recorrência** (sem simulação)
3. **Fatores Sistemáticos**: Incorpora a incerteza nas taxas de default via **volatilidade de taxa** e **análise setorial**
4. **Contribuições de Risco Aditivas**: Decompõe o risco de portfólio entre obrigadores de forma aditiva
5. **Escalabilidade**: Eficiente computacionalmente, permitindo análise de portfólios com milhares de exposições

## Componentes do Modelo

O Credit Risk+ tem **3 componentes principais**:

```
┌─────────────────────────────────────────────────────┐
│                 CREDITRISK+ MODEL                    │
├──────────────────┬──────────────────┬───────────────┤
│ CREDIT RISK      │ ECONOMIC CAPITAL │  APPLICATIONS │
│ MEASUREMENT      │ FOR CREDIT RISK  │                │
├──────────────────┼──────────────────┼───────────────┤
│ • Exposures      │ • Credit Default │ • Provisioning│
│ • Default Rates  │   Loss Distrib.  │ • Risk Limits │
│ • Volatilities   │ • Economic       │ • Portfolio   │
│ • Recovery Rates │   Analysis       │   Management  │
│ • CreditRisk+ M. │ • Scenario       │                │
│                  │   Analysis       │                │
└──────────────────┴──────────────────┴───────────────┘
```

## Tipos de Risco de Crédito

Existem **dois tipos principais** de risco de crédito:

### 1. Credit Spread Risk
- **O que é**: Risco de perda por mudança nos spreads de crédito (mudanças de preço de mercado)
- **Quando acontece**: Em portfólios mark-to-market (bonds, derivativos)
- **Abordagem**: Market risk management
- **Nota**: O Credit Risk+ **não** trata deste risco

### 2. Credit Default Risk
- **O que é**: Risco de perda por default (não-pagamento) do obrigador
- **Quando acontece**: Em qualquer exposição de crédito (loans, bonds, etc.)
- **Abordagem**: Portfolio approach + analytical techniques
- **Nota**: **Este é o foco do Credit Risk+**

## Inputs Necessários

Para aplicar o modelo, você precisa de 4 inputs principais:

### 1. **Exposições de Crédito** (E_A)
- Tamanho da exposição com cada obrigador (montante em risco)
- Exemplo: valor da linha de crédito, nominal do bond, etc.
- Considera qualquer netting bilateral e direitos de set-off

### 2. **Taxas de Default** (p_A e σ_A)
- **Taxa média** (p_A): probabilidade anual de default inferida de ratings ou mercado
- **Volatilidade** (σ_A): desvio padrão da taxa de default
- Reflete a **incerteza** nas taxas de default (não apenas Poisson puro)

#### Exemplo de Tabela de Ratings:

| Rating | Taxa Média (%) | Volatilidade (%) |
|--------|----------------|------------------|
| AAA    | 0.01           | 0.005            |
| AA     | 0.05           | 0.025            |
| A      | 0.10           | 0.050            |
| BBB    | 0.50           | 0.250            |
| BB     | 1.50           | 0.750            |
| B      | 3.00           | 1.500            |

### 3. **Taxas de Recuperação** (Recovery Rate)
- Percentual do valor em risco recuperado após default
- Exemplo: para empréstimos garantidos, ~40%; para bonds sênior, ~50%; para subordinado, ~20%
- **Perda efetiva** = Exposição × (1 - Taxa de Recuperação)

### 4. **Análise Setorial** (Sector Factors)
- Alocação de cada obrigador a **fatores sistemáticos** (setores)
- Exemplo: fatores geográficos (EUA, Japão, Europa), setores industriais, etc.
- Captura a **correlação** entre defaults de obrigadores no mesmo setor

## Horizonte de Tempo

O Credit Risk+ pode ser aplicado em **dois horizontes de tempo**:

### 1. **Horizonte de 1 Ano (Constante)**
- Apropriado para: capital econômico, provisões, limite de crédito
- Alinha com: ciclo contábil, capital regulatório
- Assumção: todas as exposições avaliadas no mesmo ponto futuro (1 ano)

### 2. **Horizonte Hold-to-Maturity (Multi-Ano)**
- Apropriado para: portfólios mantidos até vencimento (bonds de longo prazo)
- Captura: declínio de qualidade de crédito ao longo do tempo
- Estrutura a termo: taxas de default diferentes para cada ano

## Resultados do Modelo

O modelo produz:

1. **Distribuição Completa de Perdas**
   - Probabilidade de cada nível de perda
   - Permite calcular qualquer percentil (VaR), não apenas a média

2. **Perda Esperada (Expected Loss)**
   - EL = Σ (Exposição_A × Taxa de Default_A × (1 - Taxa de Recuperação))
   - Custo médio do default

3. **Capital Econômico**
   - EC = VaR(99%) - EL
   - Capital necessário para cobrir perdas inesperadas com 99% de confiança

4. **Contribuições de Risco**
   - Impacto incremental de cada obrigador no risco total
   - Usado para alocação de capital e gestão de portfólio

## Exemplo Simplificado

Suponha um portfólio com 2 obrigadores:

| Obrigador | Exposição | Taxa Default | Recuperação |
|-----------|-----------|--------------|-------------|
| A         | $1.0M     | 2%           | 40%         |
| B         | $2.0M     | 5%           | 30%         |

**Perda Esperada** (sem considerar volatilidade):
- EL_A = $1.0M × 2% × (1 - 40%) = $12k
- EL_B = $2.0M × 5% × (1 - 30%) = $70k
- **EL Total = $82k**

Mas a **distribuição** de perdas pode ser:
- Nenhum default (62%): perda = $0
- Apenas A default (2%): perda = $0.6M
- Apenas B default (5%): perda = $1.4M
- Ambos default (0.1%): perda = $2.0M

Isso resulta em:
- **Perda Esperada**: ~$82k
- **VaR 95%**: ~$1.4M
- **Capital Econômico (95%)**: $1.4M - $0.082M ≈ $1.32M

## Estrutura do Curso

Vamos aprender o Credit Risk+ em **9 notebooks**:

1. **01_introducao** (este notebook) - Conceitos e componentes
2. **02_modelo_fixo** - Modelo com taxas de default fixas (distribuição Poisson)
3. **03_modelo_variavel** - Incorporação de incerteza nas taxas (Binomial Negativa)
4. **04_exemplo_1A** - Replicar Example 1A (portfólio completo, 1 setor)
5. **05_exemplo_1B** - Example 1B (análise de movimentação de portfólio)
6. **06_exemplo_1C** - Example 1C (multi-ano, estrutura a termo)
7. **07_exemplo_2** - Example 2 (3 setores geográficos)
8. **08_exemplo_3** - Example 3 (4 setores com alocação fracionária)
9. **09_aplicacoes** - Aplicações: provisioning, limites, gestão de portfólio

## Referências

- **Documento Principal**: CreditRisk+ - A Credit Risk Management Framework (Credit Suisse First Boston, 1997)
- **Seções de Interesse**:
  - Apêndice A: Formulação matemática completa
  - Seção 3: O modelo Credit Risk+
  - Seção 4: Capital econômico
  - Seção 5: Aplicações

Vamos começar! 🚀

## Próximos Passos

No próximo notebook (02_modelo_fixo), vamos implementar o caso mais simples: **taxas de default fixas** e ver como calcular a distribuição de perdas usando a distribuição Poisson.